# Athena emulator — governance

You're the platform admin for this Athena footprint: provision
workgroups with cost and security guardrails, register the data
catalogs teams query through, tag resources for cost allocation, and
inventory what the service ships. Each admin action runs against the
live stack and records a measured capability row.

Stack (`docker-compose.yml`, network `mlops_net`): `athena`
(emulator) · `trino` (query engine) · `moto` (S3/Glue) · `jupyterlab`
(this notebook). Per ADR-0002, only Athena traffic goes to the
emulator; S3/Glue calls must still land on moto.


In [1]:
import os

# Compose sets these for in-container runs; the localhost fallbacks let the
# same notebook run against the published ports from the host.
os.environ.setdefault("AWS_ENDPOINT_URL", "http://localhost:5000")
os.environ.setdefault("AWS_ENDPOINT_URL_ATHENA", "http://localhost:5001")
os.environ.setdefault("AWS_ACCESS_KEY_ID", "test")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "test")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")

ATHENA_URL = os.environ["AWS_ENDPOINT_URL_ATHENA"]
EDGE_URL = os.environ["AWS_ENDPOINT_URL"]

print(f"athena → {ATHENA_URL}")
print(f"s3/glue → {EDGE_URL}")

athena → http://localhost:5001
s3/glue → http://localhost:5000


In [2]:
import time
import uuid
from dataclasses import dataclass

import boto3
from _evidence import persist, record, render
from _helpers import (
    client_error_status,
    delete_run_buckets,
    delete_run_catalogs,
    delete_run_databases,
    delete_workgroup_contents,
    error_code_status,
    object_keys_under,
    query_execution,
    run_workgroup_names,
)
from botocore.exceptions import ClientError

athena = boto3.client("athena")
s3 = boto3.client("s3")
glue = boto3.client("glue")

assert athena.meta.endpoint_url.rstrip("/") == ATHENA_URL, (
    athena.meta.endpoint_url
)
assert s3.meta.endpoint_url.rstrip("/") == EDGE_URL, s3.meta.endpoint_url
assert glue.meta.endpoint_url.rstrip("/") == EDGE_URL, glue.meta.endpoint_url

REGION = os.environ["AWS_DEFAULT_REGION"]
ACCOUNT_ID = "123456789012"  # accepted but not enforced by the emulator
RUN_SUFFIX = uuid.uuid4().hex[:8]
# Every resource this run creates carries the nb05- prefix (nb05_ for Glue
# databases, where '-' is invalid) so cleanup can discover leftovers — its
# own and any failed run's — instead of tracking them in a list.
ATHENA_RESOURCE_PREFIX = "nb05-"
DATABASE_PREFIX = "nb05_"

bucket = f"{ATHENA_RESOURCE_PREFIX}{uuid.uuid4().hex[:12]}"
database = f"{DATABASE_PREFIX}{RUN_SUFFIX}"
crud_workgroup = f"{ATHENA_RESOURCE_PREFIX}wg-{RUN_SUFFIX}"
enforced_workgroup = f"{ATHENA_RESOURCE_PREFIX}enf-{RUN_SUFFIX}"
fallback_workgroup = f"{ATHENA_RESOURCE_PREFIX}fb-{RUN_SUFFIX}"
managed_workgroup = f"{ATHENA_RESOURCE_PREFIX}mgr-{RUN_SUFFIX}"
disabled_workgroup = f"{ATHENA_RESOURCE_PREFIX}dis-{RUN_SUFFIX}"
encrypted_workgroup = f"{ATHENA_RESOURCE_PREFIX}enc-{RUN_SUFFIX}"
catalog_name = f"{ATHENA_RESOURCE_PREFIX}cat-{RUN_SUFFIX}"
disposable_catalog = f"{ATHENA_RESOURCE_PREFIX}del-{RUN_SUFFIX}"

s3.create_bucket(Bucket=bucket)
glue.create_database(DatabaseInput={"Name": database})


@dataclass(frozen=True)
class QueryRun:
    """Terminal outcome of one StartQueryExecution → poll cycle."""

    query_id: str
    state: str
    latency_ms: float
    execution: dict[str, object]


def workgroup(name: str) -> dict[str, object]:
    """The GetWorkGroup ``WorkGroup`` member for ``name``."""
    return athena.get_work_group(WorkGroup=name)["WorkGroup"]


def data_catalog(name: str) -> dict[str, object]:
    """The GetDataCatalog ``DataCatalog`` member for ``name``."""
    return athena.get_data_catalog(Name=name)["DataCatalog"]


def resource_arn(resource_type: str, resource_name: str) -> str:
    """``arn:aws:athena:<region>:<account>:<type>/<name>`` — the tag ARN."""
    return (
        f"arn:aws:athena:{REGION}:{ACCOUNT_ID}:{resource_type}/{resource_name}"
    )


def run_select_to_terminal(
    workgroup_name: str, output_location: str | None
) -> QueryRun:
    """Start `SELECT 1` on ``workgroup_name`` and poll to a terminal state."""
    request: dict[str, object] = {
        "QueryString": "SELECT 1 AS one",
        "QueryExecutionContext": {"Database": database},
        "WorkGroup": workgroup_name,
    }
    if output_location is not None:
        request["ResultConfiguration"] = {"OutputLocation": output_location}
    started = time.monotonic()
    query_id = athena.start_query_execution(**request)["QueryExecutionId"]
    execution = query_execution(athena, query_id)
    while execution["Status"]["State"] in {"QUEUED", "RUNNING"}:
        time.sleep(0.25)
        execution = query_execution(athena, query_id)
    return QueryRun(
        query_id=query_id,
        state=execution["Status"]["State"],
        latency_ms=(time.monotonic() - started) * 1000,
        execution=execution,
    )


def attempted_select(
    workgroup_name: str, output_location: str | None
) -> QueryRun | tuple[str, int]:
    """`SELECT 1` that may be rejected: QueryRun, or (code, status)."""
    try:
        return run_select_to_terminal(workgroup_name, output_location)
    except ClientError as exc:
        return error_code_status(exc)


print(f"bucket={bucket} database={database} suffix={RUN_SUFFIX}")

bucket=nb05-69f220db6c6c database=nb05_151e60a6 suffix=151e60a6


## What the service ships

`primary` exists out of the box (`ENABLED`, non-enforcing, engine
`AUTO`→`Athena engine version 3`) and `ListEngineVersions` reports the
single engine on offer — the inventory an admin reads first.


In [3]:
primary = workgroup("primary")
primary_config = primary["Configuration"]

assert primary["State"] == "ENABLED"
# Hard index, same as wrangler's workgroup resolution — a missing member
# would crash a consumer, so presence is part of the assertion.
assert primary_config["EnforceWorkGroupConfiguration"] is False
assert primary_config["EngineVersion"] == {
    "SelectedEngineVersion": "AUTO",
    "EffectiveEngineVersion": "Athena engine version 3",
}

engine_versions = athena.list_engine_versions()["EngineVersions"]
assert engine_versions == [
    {
        "SelectedEngineVersion": "AUTO",
        "EffectiveEngineVersion": "Athena engine version 3",
    }
]

record(
    "primary workgroup seeded (ENABLED, enforce=False, engine AUTO/3)",
    "PASS",
    f"state={primary['State']} "
    f"enforce={primary_config['EnforceWorkGroupConfiguration']}",
)
record(
    "list_engine_versions (single AUTO/engine-3 entry)",
    "PASS",
    f"{len(engine_versions)} version(s): "
    f"{engine_versions[0]['EffectiveEngineVersion']}",
)

Evidence(feature='list_engine_versions (single AUTO/engine-3 entry)', status='PASS', detail='1 version(s): Athena engine version 3', latency_ms=None)

## Provision a workgroup

`CreateWorkGroup` applies service defaults (enforce=True, engine `AUTO`)
over the admin's `Configuration`; `ConfigurationUpdates` retunes the
cost guardrail and result location in place.


In [4]:
athena.create_work_group(
    Name=crud_workgroup,
    Description="nb05 governance probe",
    Configuration={"BytesScannedCutoffPerQuery": 10485760},
    Tags=[{"Key": "project", "Value": "nb05"}],
)
created = workgroup(crud_workgroup)
created_config = created["Configuration"]

assert created_config["EnforceWorkGroupConfiguration"] is True
assert (
    created_config["EngineVersion"]["EffectiveEngineVersion"]
    == "Athena engine version 3"
)
assert created_config["BytesScannedCutoffPerQuery"] == 10485760
assert created["Description"] == "nb05 governance probe"

athena.update_work_group(
    WorkGroup=crud_workgroup,
    Description="nb05 updated",
    ConfigurationUpdates={
        "BytesScannedCutoffPerQuery": 20971520,
        "ResultConfigurationUpdates": {
            "OutputLocation": f"s3://{bucket}/wg-crud/"
        },
    },
)
updated = workgroup(crud_workgroup)
updated_config = updated["Configuration"]

assert updated["Description"] == "nb05 updated"
assert updated_config["BytesScannedCutoffPerQuery"] == 20971520
assert (
    updated_config["ResultConfiguration"]["OutputLocation"]
    == f"s3://{bucket}/wg-crud/"
)

athena.update_work_group(
    WorkGroup=crud_workgroup,
    ConfigurationUpdates={"RemoveBytesScannedCutoffPerQuery": True},
)
assert (
    "BytesScannedCutoffPerQuery"
    not in workgroup(crud_workgroup)["Configuration"]
)

listed_names = {
    summary["Name"] for summary in athena.list_work_groups()["WorkGroups"]
}
assert {"primary", crud_workgroup} <= listed_names

record(
    "workgroup create/get/update/list (defaults applied)",
    "PASS",
    f"{crud_workgroup}: enforce=True default, BytesScannedCutoffPerQuery "
    "10485760→20971520→removed via ConfigurationUpdates",
)

Evidence(feature='workgroup create/get/update/list (defaults applied)', status='PASS', detail='nb05-wg-151e60a6: enforce=True default, BytesScannedCutoffPerQuery 10485760→20971520→removed via ConfigurationUpdates', latency_ms=None)

## Names stay unique, `primary` stays put

Duplicate creates, missing lookups, and deleting the seeded `primary`
all answer `InvalidRequestException` 400 — the model declares no
`ResourceNotFoundException` for workgroup ops.


In [5]:
duplicate_error = client_error_status(
    lambda: athena.create_work_group(Name=crud_workgroup)
)
missing_error = client_error_status(
    lambda: workgroup(f"{ATHENA_RESOURCE_PREFIX}no-such-wg")
)
primary_delete_error = client_error_status(
    lambda: athena.delete_work_group(WorkGroup="primary")
)

assert duplicate_error == ("InvalidRequestException", 400)
assert missing_error == ("InvalidRequestException", 400)
assert primary_delete_error == ("InvalidRequestException", 400)

record(
    "workgroup error taxonomy (duplicate/missing/primary-delete → 400)",
    "PASS",
    "InvalidRequestException 400 for all three",
)

Evidence(feature='workgroup error taxonomy (duplicate/missing/primary-delete → 400)', status='PASS', detail='InvalidRequestException 400 for all three', latency_ms=None)

## Enforced result location

An enforced workgroup's `ResultConfiguration.OutputLocation` overrides
whatever the request passes — the admin decides where every artifact
lands; `GetQueryExecution` reports the resolved `{prefix}{QueryID}.csv`.


In [6]:
athena.create_work_group(
    Name=enforced_workgroup,
    Configuration={
        "ResultConfiguration": {
            "OutputLocation": f"s3://{bucket}/wg-enforced/"
        },
        "EnforceWorkGroupConfiguration": True,
    },
)
enforced_run = run_select_to_terminal(
    enforced_workgroup, f"s3://{bucket}/client-prefix/"
)
assert enforced_run.state == "SUCCEEDED"

enforced_location = enforced_run.execution["ResultConfiguration"][
    "OutputLocation"
]
assert (
    enforced_location
    == f"s3://{bucket}/wg-enforced/{enforced_run.query_id}.csv"
)
assert f"wg-enforced/{enforced_run.query_id}.csv" in object_keys_under(
    s3, bucket, "wg-enforced/"
)
assert object_keys_under(s3, bucket, "client-prefix/") == []

record(
    "enforced workgroup overrides client OutputLocation",
    "PASS",
    f"client passed s3://{bucket}/client-prefix/; "
    f"reported+written={enforced_location}",
    latency_ms=enforced_run.latency_ms,
)

Evidence(feature='enforced workgroup overrides client OutputLocation', status='PASS', detail='client passed s3://nb05-69f220db6c6c/client-prefix/; reported+written=s3://nb05-69f220db6c6c/wg-enforced/19243d58-4d35-463b-8aea-40ac30dbe700.csv', latency_ms=290.1450990466401)

## Fallback result location

A non-enforced workgroup's `OutputLocation` is a fallback — used only
when the request omits `ResultConfiguration`, the same request→workgroup
resolution order wrangler mirrors.


In [7]:
athena.create_work_group(
    Name=fallback_workgroup,
    Configuration={
        "ResultConfiguration": {
            "OutputLocation": f"s3://{bucket}/wg-fallback/"
        },
        "EnforceWorkGroupConfiguration": False,
    },
)
fallback_run = run_select_to_terminal(fallback_workgroup, None)
assert fallback_run.state == "SUCCEEDED"

fallback_location = fallback_run.execution["ResultConfiguration"][
    "OutputLocation"
]
assert (
    fallback_location
    == f"s3://{bucket}/wg-fallback/{fallback_run.query_id}.csv"
)
assert f"wg-fallback/{fallback_run.query_id}.csv" in object_keys_under(
    s3, bucket, "wg-fallback/"
)

record(
    "non-enforced workgroup OutputLocation fallback",
    "PASS",
    "request omitted ResultConfiguration; "
    f"reported+written={fallback_location}",
    latency_ms=fallback_run.latency_ms,
)

Evidence(feature='non-enforced workgroup OutputLocation fallback', status='PASS', detail='request omitted ResultConfiguration; reported+written=s3://nb05-69f220db6c6c/wg-fallback/af02e841-5304-438b-b002-c06b626cbb24.csv', latency_ms=282.91838604491204)

## Managed query results

`ManagedQueryResultsConfiguration.Enabled=True` makes Athena own the
results: a client-passed `OutputLocation` is ignored, no artifacts hit
S3, and rows still come back inline via `GetQueryResults` (ADR-0011).


In [8]:
athena.create_work_group(
    Name=managed_workgroup,
    Configuration={"ManagedQueryResultsConfiguration": {"Enabled": True}},
)
managed_run = run_select_to_terminal(
    managed_workgroup, f"s3://{bucket}/ignored/"
)
assert managed_run.state == "SUCCEEDED"

managed_config = managed_run.execution.get("ResultConfiguration", {})
assert "OutputLocation" not in managed_config

managed_rows = athena.get_query_results(QueryExecutionId=managed_run.query_id)[
    "ResultSet"
]["Rows"]
assert managed_rows[1]["Data"][0]["VarCharValue"] == "1"
assert object_keys_under(s3, bucket, "ignored/") == []

record(
    "managed-results workgroup (no OutputLocation, inline rows, no S3 writes)",
    "PASS",
    f"ResultConfiguration={managed_config}; "
    f"rows={len(managed_rows) - 1}; ignored/ empty",
    latency_ms=managed_run.latency_ms,
)

Evidence(feature='managed-results workgroup (no OutputLocation, inline rows, no S3 writes)', status='PASS', detail='ResultConfiguration={}; rows=1; ignored/ empty', latency_ms=288.014930090867)

## Disabled workgroups refuse work

`UpdateWorkGroup` can flip a workgroup to `DISABLED`; real Athena then
rejects `StartQueryExecution` at submit — measured here, a rejection is
`PASS`, an executed query is a `GAP` for triage.


In [9]:
feature = "disabled workgroup rejects StartQueryExecution"

athena.create_work_group(Name=disabled_workgroup)
athena.update_work_group(WorkGroup=disabled_workgroup, State="DISABLED")
assert workgroup(disabled_workgroup)["State"] == "DISABLED"

disabled_outcome = attempted_select(
    disabled_workgroup, f"s3://{bucket}/disabled/"
)
if isinstance(disabled_outcome, QueryRun):
    record(
        feature,
        "GAP",
        f"{disabled_outcome.query_id} reached {disabled_outcome.state} "
        "on a DISABLED workgroup; real AWS rejects at submit",
    )
if not isinstance(disabled_outcome, QueryRun):
    error_code, http_status = disabled_outcome
    record(feature, "PASS", f"{error_code} ({http_status})")

## Encrypt results at rest

`ResultConfiguration.EncryptionConfiguration` on the workgroup pins the
default results-encryption policy — `GetWorkGroup` echoes the stored
`EncryptionOption`/`KmsKey` verbatim for audit.


In [10]:
kms_key = (
    "arn:aws:kms:us-east-1:123456789012:"
    "key/00000000-0000-0000-0000-000000000000"
)
encryption = {"EncryptionOption": "SSE_KMS", "KmsKey": kms_key}
athena.create_work_group(
    Name=encrypted_workgroup,
    Configuration={
        "ResultConfiguration": {
            "OutputLocation": f"s3://{bucket}/wg-encrypted/",
            "EncryptionConfiguration": encryption,
        }
    },
)

stored_config = workgroup(encrypted_workgroup)["Configuration"]
stored_encryption = stored_config["ResultConfiguration"][
    "EncryptionConfiguration"
]
assert stored_encryption == encryption

record(
    "workgroup EncryptionConfiguration echoed verbatim on GetWorkGroup",
    "PASS",
    f"SSE_KMS + KmsKey echoed: {stored_encryption}",
)

Evidence(feature='workgroup EncryptionConfiguration echoed verbatim on GetWorkGroup', status='PASS', detail="SSE_KMS + KmsKey echoed: {'EncryptionOption': 'SSE_KMS', 'KmsKey': 'arn:aws:kms:us-east-1:123456789012:key/00000000-0000-0000-0000-000000000000'}", latency_ms=None)

## Data catalogs

`AwsDataCatalog` (GLUE) is pre-registered like real AWS; a LAMBDA
catalog's `Parameters.function` normalizes into `catalog` +
`metadata-function` + `record-function`. `FEDERATED` is rejected loudly
(400); missing catalogs answer 400.


In [11]:
seeded_catalog = data_catalog("AwsDataCatalog")
assert seeded_catalog["Type"] == "GLUE"
listed_catalog_names = {
    summary["CatalogName"]
    for summary in athena.list_data_catalogs()["DataCatalogsSummary"]
}
assert "AwsDataCatalog" in listed_catalog_names

lambda_function_arn = f"arn:aws:lambda:{REGION}:{ACCOUNT_ID}:function:nb05"
athena.create_data_catalog(
    Name=catalog_name,
    Type="LAMBDA",
    Description="nb05 catalog",
    Parameters={"function": lambda_function_arn},
)
fetched_catalog = data_catalog(catalog_name)
assert fetched_catalog["Parameters"]["catalog"] == catalog_name
assert fetched_catalog["Parameters"]["metadata-function"] == (
    lambda_function_arn
)
assert fetched_catalog["Parameters"]["record-function"] == (
    lambda_function_arn
)

athena.update_data_catalog(
    Name=catalog_name,
    Type="LAMBDA",
    Description="nb05 catalog updated",
    Parameters={"function": lambda_function_arn},
)
assert data_catalog(catalog_name)["Description"] == "nb05 catalog updated"

athena.create_data_catalog(
    Name=disposable_catalog,
    Type="HIVE",
    Parameters={"metadata-function": lambda_function_arn},
)
deleted_catalog = athena.delete_data_catalog(Name=disposable_catalog)[
    "DataCatalog"
]
assert deleted_catalog["Name"] == disposable_catalog
assert client_error_status(lambda: data_catalog(disposable_catalog)) == (
    "InvalidRequestException",
    400,
)

federated_error = client_error_status(
    lambda: athena.create_data_catalog(
        Name=f"{catalog_name}fed", Type="FEDERATED"
    )
)
assert federated_error == ("InvalidRequestException", 400)

record(
    "data catalogs CRUD + AwsDataCatalog seed + LAMBDA normalization",
    "PASS",
    "AwsDataCatalog=GLUE; function→catalog/metadata-function/"
    "record-function; FEDERATED→400; missing→400",
)

Evidence(feature='data catalogs CRUD + AwsDataCatalog seed + LAMBDA normalization', status='PASS', detail='AwsDataCatalog=GLUE; function→catalog/metadata-function/record-function; FEDERATED→400; missing→400', latency_ms=None)

## Tags for cost allocation

Resources are addressed by Athena ARN (`…:workgroup/`, `…:datacatalog/`);
the create-time tag is already on the workgroup, and `TagResource`/
`UntagResource`/`ListTagsForResource` mutate from there.


In [12]:
workgroup_arn = resource_arn("workgroup", crud_workgroup)
catalog_arn = resource_arn("datacatalog", catalog_name)

initial_tags = athena.list_tags_for_resource(ResourceARN=workgroup_arn)["Tags"]
assert {"Key": "project", "Value": "nb05"} in initial_tags

athena.tag_resource(
    ResourceARN=workgroup_arn, Tags=[{"Key": "owner", "Value": "nb05"}]
)
tagged_keys = {
    tag["Key"]
    for tag in athena.list_tags_for_resource(ResourceARN=workgroup_arn)["Tags"]
}
assert tagged_keys == {"project", "owner"}

athena.untag_resource(ResourceARN=workgroup_arn, TagKeys=["project"])
remaining_keys = {
    tag["Key"]
    for tag in athena.list_tags_for_resource(ResourceARN=workgroup_arn)["Tags"]
}
assert remaining_keys == {"owner"}

athena.tag_resource(
    ResourceARN=catalog_arn, Tags=[{"Key": "kind", "Value": "parity"}]
)
assert athena.list_tags_for_resource(ResourceARN=catalog_arn)["Tags"] == [
    {"Key": "kind", "Value": "parity"}
]

record(
    "tags tag/untag/list on workgroup + datacatalog ARNs",
    "PASS",
    "create-time tag present; +owner/-project on workgroup; "
    "kind=parity on datacatalog",
)

Evidence(feature='tags tag/untag/list on workgroup + datacatalog ARNs', status='PASS', detail='create-time tag present; +owner/-project on workgroup; kind=parity on datacatalog', latency_ms=None)

## Tag error taxonomy

Unknown resources answer `ResourceNotFoundException` 404; malformed
ARNs and unmodelled resource types answer `InvalidRequestException` 400.


In [13]:
unknown_resource_error = client_error_status(
    lambda: athena.list_tags_for_resource(
        ResourceARN=resource_arn(
            "workgroup", f"{ATHENA_RESOURCE_PREFIX}no-such"
        )
    )
)
malformed_arn_error = client_error_status(
    lambda: athena.tag_resource(
        ResourceARN="not-an-arn",
        Tags=[{"Key": "k", "Value": "v"}],
    )
)
untaggable_type_error = client_error_status(
    lambda: athena.list_tags_for_resource(
        ResourceARN=resource_arn("preparedstatement", "x")
    )
)

assert unknown_resource_error == ("ResourceNotFoundException", 404)
assert malformed_arn_error == ("InvalidRequestException", 400)
assert untaggable_type_error == ("InvalidRequestException", 400)

record(
    "tag error taxonomy (unknown → 404, malformed/non-taggable → 400)",
    "PASS",
    "RNFE 404 for unknown workgroup ARN; "
    "InvalidRequestException for bad ARN and preparedstatement type",
)

Evidence(feature='tag error taxonomy (unknown → 404, malformed/non-taggable → 400)', status='PASS', detail='RNFE 404 for unknown workgroup ARN; InvalidRequestException for bad ARN and preparedstatement type', latency_ms=None)

## Paging the inventory

`ListWorkGroups` honors `MaxResults`/`NextToken` — an admin pages the
fleet instead of pulling it whole.


In [14]:
feature = "list_work_groups honors MaxResults/NextToken"

paged_listing = athena.list_work_groups(MaxResults=1)
page_size_respected = (
    "NextToken" in paged_listing and len(paged_listing["WorkGroups"]) == 1
)
if page_size_respected:
    record(
        feature,
        "PASS",
        f"1 workgroup + NextToken={paged_listing['NextToken']!r}",
    )
if not page_size_respected:
    record(
        feature,
        "GAP",
        f"MaxResults=1 returned {len(paged_listing['WorkGroups'])} "
        f"workgroups, NextToken={'NextToken' in paged_listing}",
    )

## Capabilities demonstrated

Every feature above recorded a measured row. The matrix persists to
`parity/05_governance.md`; `PARITY.md` at the project root is
regenerated from all fragments.


In [15]:
print(render())
persist("05_governance")

| feature | status | detail | latency_ms |
|---|---|---|---|
| primary workgroup seeded (ENABLED, enforce=False, engine AUTO/3) | PASS | state=ENABLED enforce=False |  |
| list_engine_versions (single AUTO/engine-3 entry) | PASS | 1 version(s): Athena engine version 3 |  |
| workgroup create/get/update/list (defaults applied) | PASS | nb05-wg-151e60a6: enforce=True default, BytesScannedCutoffPerQuery 10485760→20971520→removed via ConfigurationUpdates |  |
| workgroup error taxonomy (duplicate/missing/primary-delete → 400) | PASS | InvalidRequestException 400 for all three |  |
| enforced workgroup overrides client OutputLocation | PASS | client passed s3://nb05-69f220db6c6c/client-prefix/; reported+written=s3://nb05-69f220db6c6c/wg-enforced/19243d58-4d35-463b-8aea-40ac30dbe700.csv | 290.145 |
| non-enforced workgroup OutputLocation fallback | PASS | request omitted ResultConfiguration; reported+written=s3://nb05-69f220db6c6c/wg-fallback/af02e841-5304-438b-b002-c06b626cbb24.csv | 282.91

PosixPath('/home/gabriel-menezes/Documents/repos/mlops/projects/athena_emulator/parity/05_governance.md')

## Clean up

The run's workgroups, catalogs, bucket, and Glue database are throwaway
— delete them so the stack is left as you found it.


In [16]:
for name in run_workgroup_names(athena, ATHENA_RESOURCE_PREFIX):
    delete_workgroup_contents(athena, name)
    athena.delete_work_group(WorkGroup=name)
delete_run_catalogs(athena, ATHENA_RESOURCE_PREFIX)
delete_run_buckets(s3, ATHENA_RESOURCE_PREFIX)
delete_run_databases(glue, DATABASE_PREFIX)
print("cleaned: nb05 workgroups + catalogs + buckets + databases removed")

cleaned: nb05 workgroups + catalogs + buckets + databases removed
